# AfriVoices — v10 UNSCRIPTED (exploration DOCUMENTÉE, non retenue)

**Verdict final : gain dev non transféré au leaderboard (+0.031 vs 0.39923) — modèle
non retenu.** Ce notebook est conservé parce qu'il documente deux résultats importants
du RAPPORT (Partie II §3) :

1. **Le filtre d'entraînement `durée ≤ 20s` excluait 85-94 % de la parole spontanée**
   (médianes 41-70s selon la langue) — le modèle n'avait presque jamais entendu le
   registre qui compose jusqu'à 42 % du test.
2. **L'alignement forcé CTC** (§5.2) convertit les clips longs en segments ≤ 18s avec
   leur texte (< 1 % de rejets) : ressource spontanée ×25 (kln : 182 clips → 4 973
   segments). Le levier données est réel ; sa validation exige un dev multi-locuteurs
   (le dev anvke n'a que 2-6 voix par case — d'où le non-transfert).

Mix : ~74 % spontané ciblé + 16 % lu anti-oubli + 10 % swa. Départ v9-2 → v10-unscr.
Ordre : §1 (install → REDÉMARRER → §1) → §2 → ... → §5.2 (alignement) → §6 → §7 → §8.

## 1. Dépendances (redémarrer le runtime après la 1ère exécution)

In [ ]:
import importlib, subprocess, sys
need = [m for m in ["kenlm", "pyctcdecode", "jiwer", "datasets", "accelerate"]
        if importlib.util.find_spec(m) is None]
if need:
    print("installation de", need, "...")
    if "pyctcdecode" in need:
        # pyctcdecode épingle numpy<2 -> --no-deps pour préserver le numpy de Colab
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "pyctcdecode", "pygtrie"], check=False)
    for m in need:
        if m == "kenlm":
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "https://github.com/kpu/kenlm/archive/master.zip"], check=False)
        elif m != "pyctcdecode":
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", m], check=False)
    print("⚠️ REDÉMARRE le runtime (Exécution > Redémarrer la session), puis relance cette cellule.")
else:
    import numpy
    print(f"✓ dépendances présentes (numpy {numpy.__version__}, intact) — continue en §2")

## 2. Config — RÈGLE `TRANCHE` ICI (0, 1, ... une par session)

In [ ]:
import os, shutil
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["HF_DATASETS_CACHE"] = "/content/hf_cache/datasets"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

from google.colab import drive
drive.mount("/content/drive")
import numpy as np, torch

# ========== RÉGLAGES DE SESSION ==========
TRANCHE = 0                                  # 0 puis 1, ... (une par session)
NB_TRANCHES = {"unscripted": 6, "scripted": 8, "swa": 6}   # 1/6 spontané par session (contrainte disque mesurée)
CAP_EPOCH = 40000                            # sonde ~2500 pas (mettre None pour une époque complète)
# =========================================

BASE = "/content/drive/MyDrive/afrivoices"
START_DIR = f"{BASE}/baobab-asr-v9-2"        # départ = meilleur modèle actuel (LB 0.40283)
OUT_DIR   = f"{BASE}/baobab-asr-v10-unscr"
LM        = f"{BASE}/lm_v2"

SAMPLING_RATE = 16_000; MAX_AUDIO_S = 20; SEED = 210 + TRANCHE
os.makedirs(OUT_DIR, exist_ok=True); torch.manual_seed(SEED); np.random.seed(SEED)

# MIX PAR CASE : ~74% spontané ciblé / ~16% lu anti-oubli / 10% swa replay
probs = {
    "kln_unscripted": 0.20, "mas_unscripted": 0.17, "som_unscripted": 0.16,
    "kik_unscripted": 0.13, "luo_unscripted": 0.08,
    "kln_scripted": 0.04, "mas_scripted": 0.04, "som_scripted": 0.03,
    "kik_scripted": 0.03, "luo_scripted": 0.02,
    "swa": 0.10,
}
assert abs(sum(probs.values()) - 1.0) < 1e-6

# parts de spontané par langue dans le TEST (comptage exact P0b §4) — pour le macro pondéré
P_UNSCR_TEST = {"swa": 1.0, "som": 0.231, "kik": 0.186, "luo": 0.212, "kln": 0.385, "mas": 0.422}

if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0); print(f"GPU {p.name} {p.total_memory/1e9:.0f} Go")
else:
    print("⚠️ pas de GPU !")
t, u, f = shutil.disk_usage("/content"); print(f"Disque libre : {f/1e9:.0f} Go")
assert os.path.isdir(START_DIR), f"modèle de départ absent : {START_DIR}"
assert os.path.isdir(LM), f"LM absent : {LM}"
print("mix par case :", probs)

## 3. Processor (vocab figé, depuis le modèle de départ) + clean_text

In [ ]:
import re
from transformers import Wav2Vec2BertProcessor

processor = Wav2Vec2BertProcessor.from_pretrained(START_DIR)
tokenizer = processor.tokenizer
print("vocab:", tokenizer.vocab_size, "| pad:", tokenizer.pad_token)

def clean_text(t):
    t = (t or "").lower().strip()
    t = re.sub(r"[^\w\s\u0129\u0169\u00e1\u00e0\u00e2\u00e4\u00e9\u00e8\u00ea\u00eb\u00ed\u00ec\u00ee\u00ef\u00f3\u00f2\u00f4\u00f6\u00fa\u00f9\u00fb\u00fc\']", "", t)
    return re.sub(r"\s+", " ", t)

## 4. Répartition Anv-ke par (langue × TYPE) : tranche spontanée + replay lu

Les noms de parquets portent scripted/unscripted. Le spontané est la ressource rare :
sa tranche prend 1/2 des fichiers (`NB_TRANCHES["unscripted"]=2`), le lu 1/8 (replay).

In [ ]:
import glob
ANVKE = f"{BASE}/anvke"
assert os.path.isdir(ANVKE), f"introuvable: {ANVKE}"
print("dossiers vus dans anvke:", os.listdir(ANVKE))

ALIAS = {"kik": ["kik", "kikuyu", "gikuyu"], "luo": ["luo", "dholuo"], "kln": ["kln", "kalenjin"],
         "mas": ["mas", "maasai"], "som": ["som", "somali"]}
lang_dirs = {}
for d in os.listdir(ANVKE):
    dl = d.lower()
    for lang, als in ALIAS.items():
        if any(a in dl for a in als) and lang not in lang_dirs:
            lang_dirs[lang] = os.path.join(ANVKE, d)
assert len(lang_dirs) == 5, f"seulement {len(lang_dirs)}/5 dossiers reconnus"
KE_LANGS = list(lang_dirs.keys())

def est_unscr(p): return "unscripted" in os.path.basename(p).lower()

train_files, dev_files = {}, {}
for lang, root in lang_dirs.items():
    allp = sorted(glob.glob(f"{root}/**/*.parquet", recursive=True))
    tr = [p for p in allp if "/train/" in p or os.path.basename(p).startswith("train")]
    dv = [p for p in allp if "/dev/" in p or os.path.basename(p).startswith("dev")]
    if not tr and not dv: tr = allp
    for typ in ("unscripted", "scripted"):
        pool = [p for p in tr if est_unscr(p) == (typ == "unscripted")]
        nb = NB_TRANCHES[typ]
        train_files[(lang, typ)] = [p for i, p in enumerate(pool) if i % nb == (TRANCHE % nb)]
        dev_files[(lang, typ)] = [p for p in dv if est_unscr(p) == (typ == "unscripted")]
        print(f"{lang}/{typ:10}: {len(pool):>3} parquets train ({len(train_files[(lang, typ)])} dans la tranche) | {len(dev_files[(lang, typ)])} dev")
    if not train_files[(lang, "unscripted")]:
        print(f"  ⚠️ {lang}: AUCUN parquet train unscripted — la case sera absente du mix (probs renormalisés)")

### 4.0 — Filet anti-fuite : aucun clip du TEST dans ces données ?

In [ ]:
import pandas as pd, glob as g
test_ids = set()
for pat in [f"{BASE}/test/**/*.parquet", "/content/test_data/**/*.parquet"]:
    for pq in g.glob(pat, recursive=True):
        try: test_ids.update(pd.read_parquet(pq, columns=["id"])["id"].astype(str))
        except Exception: pass
print("ids test chargés:", len(test_ids))
if not test_ids:
    print("test introuvable localement — télécharge-le pour vérifier (recommandé)")
else:
    overlap = 0; seen = 0
    for key, files in train_files.items():
        for f_ in files[:2]:
            try:
                df = pd.read_parquet(f_, columns=["filename"])
                ids = set(df["filename"].astype(str).str.replace(".wav", "", regex=False)); seen += len(ids)
                overlap += len(ids & test_ids)
            except Exception as e:
                print("skip", os.path.basename(f_), str(e)[:50])
    print(f"chevauchement TEST: {overlap}/{seen} clips échantillonnés")
    assert overlap == 0, "FUITE DÉTECTÉE — ne pas entraîner, signaler aux organisateurs"
    print("OK : aucun chevauchement détecté sur l'échantillon")

### 4.1 — Éval STRATIFIÉE fixe par (langue × type) — l'instrument de la session

40 clips/case, ≤20s (l'éval définitive avec clips longs reste P0b). Sert au guard du
Trainer (WER greedy agrégé) et aux mesures LM départ/sortie (§4.3 / §8).

In [ ]:
from datasets import load_dataset, Audio, concatenate_datasets
import soundfile as sf, io, base64

N_EVAL_CASE = 40

def duree_bytes(b):
    try: i = sf.info(io.BytesIO(b)); return i.frames / i.samplerate
    except Exception:
        try: i = sf.info(io.BytesIO(base64.b64decode(b))); return i.frames / i.samplerate
        except Exception: return 999.0

def infos(ex):
    t = (ex.get("transcription") or "").strip()
    b = ex["audio"].get("bytes") if isinstance(ex["audio"], dict) else None
    dur = duree_bytes(b) if b else 999.0
    ex["ok"] = bool(t) and dur <= MAX_AUDIO_S
    ex["labels"] = tokenizer(clean_text(t)).input_ids if t else []
    return ex

eval_per_case = {}
for (lang, typ), dv in dev_files.items():
    if not dv:
        print(f"eval {lang}/{typ}: 0 parquet dev — case absente de l'instrument"); continue
    d = load_dataset("parquet", data_files=dv[:2], split="train").cast_column("audio", Audio(decode=False))
    d = d.map(infos).filter(lambda ok: ok, input_columns=["ok"])
    d = d.remove_columns([c for c in d.column_names if c not in ("audio", "labels")])
    d = d.shuffle(seed=42).select(range(min(N_EVAL_CASE, len(d))))
    eval_per_case[(lang, typ)] = d
    print(f"eval {lang}/{typ}: {len(d)}")

### 4.2 — Swahili : pool replay (prepared_v5, tranche) + éval « mixte »

In [ ]:
from datasets import Dataset, load_from_disk
SAVE_SWA = f"{BASE}/prepared_v5/swa"
assert os.path.isdir(SAVE_SWA), f"prepared_v5/swa absent : {SAVE_SWA}"
swa_ds = load_from_disk(SAVE_SWA); print("swahili rechargé:", len(swa_ds))
sp = swa_ds.train_test_split(test_size=60, seed=42)
nb = NB_TRANCHES["swa"]
swa_train_ds = sp["train"].filter(lambda ex, i: i % nb == (TRANCHE % nb), with_indices=True)
eval_per_case[("swa", "mixte")] = sp["test"].cast_column("audio", Audio(decode=False))
print("swa train (tranche):", len(swa_train_ds), "| swa eval:", len(eval_per_case[('swa', 'mixte')]))

### 4.3 — BASELINE LM du modèle de DÉPART sur ces clips exacts

Mesurée avant l'entraînement, avec KenLM v2 à (0.7, 0.5) — la §8 remesure le modèle
entraîné sur les mêmes clips : deltas propres, macro pondéré par les poids du test.
~10-15 min.

In [ ]:
import torch, numpy as np, librosa, jiwer
from transformers import Wav2Vec2BertForCTC
from pyctcdecode import build_ctcdecoder
from collections import Counter

device = "cuda" if torch.cuda.is_available() else "cpu"

raw = [t for t, _ in sorted(tokenizer.get_vocab().items(), key=lambda kv: kv[1])]
labels_dec, n_ = [], 0
for t in raw:
    if t == tokenizer.pad_token: labels_dec.append("")
    elif t in ("|", " "): labels_dec.append(" ")
    elif t in {"[UNK]", "<s>", "</s>"}: n_ += 1; labels_dec.append("\u2047" * n_)
    else: labels_dec.append(t)

def unigrams(lang, top=50000):
    c = Counter()
    with open(f"{LM}/{lang}.txt", encoding="utf-8") as f:
        for line in f: c.update(line.split())
    return [w for w, _ in c.most_common(top)]

UNI = {l: unigrams(l) for l in ["swa", "kik", "luo", "kln", "mas", "som"]}
decoders = {l: build_ctcdecoder(labels_dec, kenlm_model_path=f"{LM}/{l}.bin",
                                unigrams=UNI[l], alpha=0.7, beta=0.5) for l in UNI}

def decode_bytes(a):
    b = a.get("bytes") if isinstance(a, dict) else a
    if not b: raise ValueError("vide")
    try: arr, sr = sf.read(io.BytesIO(b), dtype="float32")
    except Exception: arr, sr = sf.read(io.BytesIO(base64.b64decode(b)), dtype="float32")
    if arr.ndim > 1: arr = arr.mean(axis=1)
    if sr != 16000: arr = librosa.resample(arr, orig_sr=sr, target_sr=16000)
    return arr.astype(np.float32)

def eval_lm(model_dir, eval_per_case):
    m = Wav2Vec2BertForCTC.from_pretrained(model_dir).to(device).eval()
    res = {}
    for (lang, typ), d in sorted(eval_per_case.items()):
        preds, refs = [], []
        with torch.no_grad():
            for j in range(len(d)):
                ex = d[j]
                arr = decode_bytes(ex["audio"])
                fe = processor.feature_extractor([arr], sampling_rate=16000, return_tensors="pt")
                lg = m(**{k: v.to(device) for k, v in fe.items()}).logits[0].float().cpu().numpy()
                preds.append(decoders[lang].decode(lg))
                lab = [t for t in ex["labels"] if t != -100]
                refs.append(tokenizer.decode(lab, group_tokens=False))
        res[(lang, typ)] = jiwer.wer(refs, preds)
        print(f"  {lang}/{typ}: {res[(lang, typ)]:.4f} (n={len(d)})")
    del m; torch.cuda.empty_cache()
    return res

def macro_pondere(wer_case):
    tot, nl = 0.0, 0
    for lang in ["swa", "som", "kik", "luo", "kln", "mas"]:
        p = P_UNSCR_TEST[lang]
        if lang == "swa":
            w = wer_case.get(("swa", "mixte"))
            if w is None: continue
            tot += w; nl += 1; continue
        wu, ws = wer_case.get((lang, "unscripted")), wer_case.get((lang, "scripted"))
        if wu is None or ws is None: continue
        tot += p * wu + (1 - p) * ws; nl += 1
    return tot / max(nl, 1), nl

print(f"=== baseline LM départ ({START_DIR.split('/')[-1]}) ===")
base_wer = eval_lm(START_DIR, eval_per_case)
base_macro, _ = macro_pondere(base_wer)
print(f"macro pondéré départ : {base_macro:.4f}")

## 5. Cache local de la tranche — par paquets de 8 parquets

Empreinte transitoire ≈ 6 Go ; l'Arrow s'accumule. **Aucun map/filter ensuite** —
le scan §5.1 fait le reste en RAM. Si le disque descend sous 15 Go : augmente
`NB_TRANCHES["unscripted"]` et recommence.

In [ ]:
from datasets import load_dataset, concatenate_datasets, Audio
import shutil

CHUNK = 8
case_ranges = {}
parts = []; offset = 0
for (lang, typ), files in train_files.items():
    if not files: continue
    key = f"{lang}_{typ}"
    case_parts = []
    for i in range(0, len(files), CHUNK):
        d = load_dataset("parquet", data_files=files[i:i + CHUNK], split="train")
        d = d.cast_column("audio", Audio(decode=False))
        case_parts.append(d)
        t, u, f = shutil.disk_usage("/content")
        print(f"  {key} paquet {i//CHUNK + 1}/{(len(files) + CHUNK - 1)//CHUNK}: +{len(d)} | libre {f/1e9:.0f} Go")
        if f / 1e9 < 15:
            raise RuntimeError("Disque < 15 Go : augmente NB_TRANCHES et recommence la tranche")
    dl = concatenate_datasets(case_parts)
    case_ranges[key] = (offset, offset + len(dl)); offset += len(dl)
    parts.append(dl)
    print(f"{key}: {len(dl)} clips")
ke_tranche = concatenate_datasets(parts)
print("\nTRANCHE", TRANCHE, ":", len(ke_tranche), "clips |", case_ranges)

### 5.1 — Scan léger (durées + labels en RAM, ZÉRO copie Arrow)

In [ ]:
ok_flags = []; all_labels = []
BS = 256
for start in range(0, len(ke_tranche), BS):
    batch = ke_tranche[start:start + BS]
    for a, t in zip(batch["audio"], batch["transcription"]):
        txt = (t or "").strip()
        b = a.get("bytes") if isinstance(a, dict) else None
        dur = duree_bytes(b) if b else 999.0
        ok = bool(txt) and dur <= MAX_AUDIO_S
        ok_flags.append(ok)
        all_labels.append(tokenizer(clean_text(txt)).input_ids if ok else [])
    if (start // BS) % 50 == 0:
        print(f"  scan {start}/{len(ke_tranche)}")
print("scan fini | gardés:", sum(ok_flags), "/", len(ok_flags))

### 5.2 — SEGMENTATION du spontané par ALIGNEMENT FORCÉ (CTC, GPU ~20-50 min)

Les clips spontanés (médianes 41-70s) sont inutilisables tels quels (filtre ≤ 20s).
Alignement forcé avec v9-2 (`torchaudio.functional.forced_align`) → position de chaque
mot → découpe aux silences entre mots en segments ≤ 18s **avec leur texte**.

In [ ]:
import torch, torchaudio, numpy as np
from transformers import Wav2Vec2BertForCTC
from tqdm.auto import tqdm

MAX_SEG_SEC, TARGET_SEC, MIN_SEG_SEC = 18.0, 14.0, 3.0
MAX_CLIP_SEC, N_CLIPS_ALIGN, MIN_SCORE, GAP_MIN = 90.0, 1200, 0.30, 0.12

am = Wav2Vec2BertForCTC.from_pretrained(START_DIR, ctc_zero_infinity=True).to(device).eval()
BLANK = tokenizer.pad_token_id
UNK   = tokenizer.unk_token_id
SPACE = tokenizer.convert_tokens_to_ids(getattr(tokenizer, "word_delimiter_token", None) or "|")

def spans_de(al, sc):
    out, i, T = [], 0, len(al)
    while i < T:
        t = int(al[i]); j = i
        while j + 1 < T and int(al[j + 1]) == t: j += 1
        if t != BLANK:
            out.append((t, i, j + 1, float(np.exp(sc[i:j + 1]).mean())))
        i = j + 1
    return out

segments = {}
for key, (a, b) in case_ranges.items():
    if not key.endswith("unscripted"): continue
    idxs = list(range(a, b)); np.random.default_rng(SEED).shuffle(idxs)
    segs, alignes, courts, rejets = [], 0, 0, 0
    for i in tqdm(idxs, desc=key, leave=False):
        if alignes >= N_CLIPS_ALIGN: break
        ex = ke_tranche[i]
        txt = clean_text((ex.get("transcription") or "").strip())
        if not txt: continue
        bts = ex["audio"].get("bytes") if isinstance(ex["audio"], dict) else None
        dur = duree_bytes(bts) if bts else 999.0
        ids = tokenizer(txt).input_ids
        if UNK in ids: rejets += 1; continue
        if dur <= 20.0:
            segs.append((i, 0, None, ids)); courts += 1; continue
        if dur > MAX_CLIP_SEC: continue
        try:
            arr = decode_bytes(ex["audio"])
            fe = processor.feature_extractor([arr], sampling_rate=16000, return_tensors="pt")
            with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
                lg = am(**{k: v.to(device) for k, v in fe.items()}).logits.float()
            logp = torch.log_softmax(lg, dim=-1)
            spf = len(arr) / logp.shape[1]
            tg = torch.tensor([ids], dtype=torch.int32, device=device)
            al, sc = torchaudio.functional.forced_align(logp, tg, blank=BLANK)
            al, sc = al[0].cpu().numpy(), sc[0].float().cpu().numpy()
        except Exception:
            torch.cuda.empty_cache(); rejets += 1; continue
        sp = spans_de(al, sc)
        if len(sp) != len(ids) or np.mean([x[3] for x in sp]) < MIN_SCORE:
            rejets += 1; continue
        mots_txt, mots, cur = txt.split(), [], []
        for (t, s0, e0, _) in sp:
            if t == SPACE:
                if cur: mots.append((cur[0][1] * spf / 16000, cur[-1][2] * spf / 16000)); cur = []
            else:
                cur.append((t, s0, e0))
        if cur: mots.append((cur[0][1] * spf / 16000, cur[-1][2] * spf / 16000))
        if len(mots) != len(mots_txt): rejets += 1; continue
        i0, k = 0, 0
        while k < len(mots):
            deb = mots[i0][0]
            if (mots[k][1] - deb >= TARGET_SEC and
                (k + 1 == len(mots) or mots[k + 1][0] - mots[k][1] >= GAP_MIN)) \
               or k + 1 == len(mots) \
               or (mots[k + 1][1] - deb > MAX_SEG_SEC):
                fin = mots[k][1]
                if fin - deb >= MIN_SEG_SEC:
                    s_e = max(0, int((deb - 0.1) * 16000)); e_e = min(len(arr), int((fin + 0.1) * 16000))
                    segs.append((i, s_e, e_e, tokenizer(" ".join(mots_txt[i0:k + 1])).input_ids))
                i0 = k + 1
            k += 1
        alignes += 1
    segments[key] = segs
    print(f"{key:16}: {len(segs):5} segments ({alignes} clips alignés, {courts} courts gardés, {rejets} rejetés)")
del am; torch.cuda.empty_cache()
print("segmentation terminée")

## 6. Wrapper : segments spontanés alignés + lu (clips ≤20s) + swa

Les groupes d'échantillonnage sont les **cases** (langue × type) ; le spontané passe
par les segments de la §5.2 (facteurs de sur-échantillonnage sains, ×1-3).

In [ ]:
import torch, numpy as np

items, groups = [], {}
for key, (a, b) in case_ranges.items():
    if key.endswith("unscripted"):
        lst = segments.get(key, [])
    else:
        lst = [(i, 0, None, all_labels[i]) for i in range(a, b) if ok_flags[i]]
    groups[key] = list(range(len(items), len(items) + len(lst)))
    items += [("ke", x) for x in lst]
groups["swa"] = list(range(len(items), len(items) + len(swa_train_ds)))
items += [("swa", k) for k in range(len(swa_train_ds))]

def build_epoch_indices(groups, probs, seed):
    rng = np.random.default_rng(seed)
    present = {l: v for l, v in groups.items() if v}
    tot = sum(len(v) for v in present.values())
    pr = {l: probs[l] for l in present}; sN = sum(pr.values()); pr = {l: p / sN for l, p in pr.items()}
    out = []
    for l, idxs in present.items():
        target = max(1, int(tot * pr[l]))
        arr = np.tile(np.array(idxs), int(np.ceil(target / len(idxs))))[:target]
        out.append(arr)
        print(f"  {l}: {len(idxs)} uniques -> {target} tirages (x{target/len(idxs):.2f})")
    allidx = np.concatenate(out); rng.shuffle(allidx)
    return allidx.tolist()

epoch_indices = build_epoch_indices(groups, probs, SEED)
if CAP_EPOCH: epoch_indices = epoch_indices[:int(CAP_EPOCH)]
print("époque tranche:", len(epoch_indices), "échantillons")

class SegWrap(torch.utils.data.Dataset):
    def __init__(self, items, idx, ke, swa):
        self.items = items; self.idx = idx; self.ke = ke; self.swa = swa
    def __len__(self): return len(self.idx)
    def __getitem__(self, n):
        kind, payload = self.items[self.idx[n]]
        if kind == "ke":
            i, s0, e0, lab = payload
            arr = decode_bytes(self.ke[i]["audio"])
            return {"array": arr[s0:(e0 if e0 is not None else len(arr))], "labels": lab}
        ex = self.swa[payload]
        return {"array": decode_bytes(ex["audio"]), "labels": ex["labels"]}

train_ds = SegWrap(items, epoch_indices, ke_tranche, swa_train_ds)
print("wrapper segments prêt :", len(train_ds), "échantillons")

## 7. Fine-tuning (1 époque cappée)

Collator **tolérant** (segments `array` ET éval `audio`), `gradient_checkpointing`
désactivé (incompatible avec le SpecAugment aléatoire au recalcul), batch 2 × accum 8
(effectif 16), éval interne coupée (le juge est la §8, avec LM), checkpoint/500 pas.

In [ ]:
import torch, gc, os, random
import numpy as np
from dataclasses import dataclass
from transformers import Wav2Vec2BertForCTC, TrainingArguments, Trainer
from transformers.trainer_utils import get_last_checkpoint

gc.collect(); torch.cuda.empty_cache()

def spec_augment_doux(f, F=8, T=10):
    f = f.copy(); Tl, D = f.shape
    a = random.randint(0, F); a0 = random.randint(0, max(0, D - a)); f[:, a0:a0 + a] = 0
    t = random.randint(0, T); t0 = random.randint(0, max(0, Tl - t)); f[t0:t0 + t, :] = 0
    return f

@dataclass
class Collator:
    processor: object
    training: bool = True
    def __call__(self, features):
        arrays = [f["array"] if "array" in f else decode_bytes(f["audio"]) for f in features]
        b = self.processor.feature_extractor(arrays, sampling_rate=16000,
                                             return_tensors="pt", padding=True)
        if self.training:
            x = b["input_features"].numpy()
            for i in range(len(x)): x[i] = spec_augment_doux(x[i])
            b["input_features"] = torch.tensor(x)
        lb = self.processor.tokenizer.pad([{"input_ids": f["labels"]} for f in features],
                                          padding=True, return_tensors="pt")
        b["labels"] = lb["input_ids"].masked_fill(lb.attention_mask.ne(1), -100)
        return b

collator_train = Collator(processor, training=True)

model = Wav2Vec2BertForCTC.from_pretrained(START_DIR, ctc_zero_infinity=True)
model.config.use_cache = False
print(f"Params {sum(p.numel() for p in model.parameters())/1e6:.0f}M — départ {START_DIR.split('/')[-1]}")
print(f"train_ds: {len(train_ds)} échantillons")

use_bf16 = torch.cuda.is_bf16_supported()
args = TrainingArguments(
    output_dir=OUT_DIR, remove_unused_columns=False,
    per_device_train_batch_size=2, gradient_accumulation_steps=8,
    gradient_checkpointing=False,
    learning_rate=1e-5, warmup_ratio=0.03, num_train_epochs=1,
    bf16=use_bf16, fp16=(not use_bf16),
    eval_strategy="no",
    save_strategy="steps", save_steps=500, save_total_limit=2,
    logging_steps=100, report_to="none", dataloader_num_workers=4,
)

trainer = Trainer(model=model, args=args, train_dataset=train_ds, data_collator=collator_train)
last = get_last_checkpoint(OUT_DIR) if os.path.isdir(OUT_DIR) else None
print("reprise depuis:", last)
trainer.train(resume_from_checkpoint=last)
trainer.save_model(OUT_DIR); processor.save_pretrained(OUT_DIR)
print("✓ sauvegardé ->", OUT_DIR)

## 8. Verdict : LM par case, mêmes clips qu'au départ, macro pondéré par le test

In [ ]:
print(f"=== sortie ({OUT_DIR.split('/')[-1]}) — LM (0.7, 0.5), mêmes clips que §4.3 ===")
new_wer = eval_lm(OUT_DIR, eval_per_case)
new_macro, nl = macro_pondere(new_wer)

print(f"\n{'case':20} {'départ':>8} {'sortie':>8} {'delta':>8}")
pire_regression_lu = 0.0
for key in sorted(base_wer):
    d = new_wer[key] - base_wer[key]
    print(f"{key[0]}/{key[1]:12} {base_wer[key]:>8.4f} {new_wer[key]:>8.4f} {d:>+8.4f}")
    if key[1] == "scripted":
        pire_regression_lu = max(pire_regression_lu, d)

delta_macro = new_macro - base_macro
print(f"\nmacro pondéré : départ {base_macro:.4f} -> sortie {new_macro:.4f} (delta {delta_macro:+.4f})")
print(f"pire régression sur une case lue : {pire_regression_lu:+.4f}")

if delta_macro <= -0.010 and pire_regression_lu <= 0.03:
    print("\n✅ ADOPTER : relance P0b avec MODEL_NAME='baobab-asr-v10-unscr' (table officielle),")
    print("   puis soumets via afrivoices_soumission_par_type en changeant UNIQUEMENT MODEL_NAME.")
    print("   Tranche suivante (TRANCHE=1) tant que le gain pondéré d'une session reste >= 0.005.")
elif delta_macro <= -0.010:
    print("\n⚠️ Gain global mais érosion du lu > +0.03 : rejoue avec plus de replay")
    print("   (ex. scripted 0.04->0.06 par langue, unscripted réduit d'autant), même TRANCHE.")
else:
    print("\n❌ Gain insuffisant sur cette tranche : n'adopte pas. Options : TRANCHE=1 (autres")
    print("   données), CAP_EPOCH plus grand si tu avais sondé court, ou arrêt du levier ici.")

## 9. Bilan de l'exploration

Sonde (40 clips/case, §8) : −0.0137 pondéré, porté par kik. Dev complet (P0b, 80
clips/case) : −0.0013. **Leaderboard : +0.031.** Diagnostic : le dev repose sur 2-6
locuteurs par case (train/dev disjoints en voix, mais trop peu de voix pour mesurer un
changement de modèle) — le fine-tuning s'est ajusté aux voix d'entraînement. Levier
conservé pour de futurs travaux avec un dev multi-locuteurs ; détails : RAPPORT
Partie II §3.